# LFW 04. Probe search and certification

## 예상 소요 시간

| 실행 모드 | 예상 시간 | 주로 오래 걸리는 구간 |
| --- | ---: | --- |
| `EXECUTE_STAGE=False` | 1초 미만 | run 연결과 자동 입력 상태 확인 |
| `EXECUTE_STAGE=True` | LFW 기준 약 1~10분 | DB 벡터 로딩, template 집계, probe×gallery 인증 |

> 00의 고정 protocol과 03의 DB 벡터를 자동으로 연결합니다. 별도 probe/template CSV 환경 변수는 선택적 override입니다. 30초마다 heartbeat를 출력합니다.

목표: registered/known-unknown/unknown-unknown probe를 검색하고, 압축 각도 오차로 accept/reject/defer 결정을 인증합니다. PCA 검색 벡터는 DB에서 256D로 유지하지만, 각도 오차 인증은 원본과 비교할 수 있도록 PCA 모델로 복원한 512D 공간에서 별도로 계산합니다.

> **재시작/재개 규칙(필수)**: 임의 셀에서 시작하지 말고 **Kernel Restart 후 Run All**을 사용합니다. 00·02·03의 최근 `completed` artifact를 검증한 뒤 새 04 attempt를 만듭니다. 중단되면 04 전체를 다시 실행하십시오. 00 protocol, 02 compressor, 03 벡터가 바뀌면 영향을 받는 상위 단계부터 다시 실행해야 합니다.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
from research.runtime import ProgressReporter, RunStore, resolve_active_run

EXECUTE_STAGE = True
RUN_ROOT = PROJECT_ROOT / 'runs' / 'lfw'
LEGACY_RUN_ROOT = PROJECT_ROOT / 'runs'
try:
    RUN_DIR = resolve_active_run(RUN_ROOT)
except FileNotFoundError:
    RUN_ROOT = LEGACY_RUN_ROOT
    RUN_DIR = resolve_active_run(RUN_ROOT)
PROGRESS = ProgressReporter('04 probe search/certification', heartbeat_seconds=30)
PROBES_CSV_VALUE = os.environ.get('RONBUN_PROBES_CSV', '').strip()
TEMPLATES_CSV_VALUE = os.environ.get('RONBUN_TEMPLATES_CSV', '').strip()


## Plan

- 00의 고정 gallery와 세 probe 유형을 자동으로 읽습니다.
- DB의 `origin_512`/`pca_256`를 protocol image path로 결합하고, 추출 실패 행을 coverage에 기록합니다.
- Gallery template과 exact fallback을 누수 없이 생성합니다.
- PCA 인증은 reconstructed 512D에서 수행하고, 256D DB 검색과 동일한 값이라고 과장하지 않습니다.
- Certified feature, 자동 생성 입력, coverage summary를 attempt artifact로 저장합니다.


In [ ]:
def attach_run(run_dir: Path) -> tuple[RunStore, dict]:
    run = RunStore.open(run_dir)
    manifest = json.loads(run.manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') == 'completed' or (run_dir / 'COMPLETED').exists():
        raise RuntimeError('Completed runs are immutable.')
    return run, manifest


def resolve_input(value: str, configured: str | None) -> Path | None:
    selected = value or (configured or '')
    if not selected:
        return None
    path = Path(selected)
    return path.resolve() if path.is_absolute() else (PROJECT_ROOT / path).resolve()


def latest_completed_attempt(run_dir: Path, phase_name: str) -> int:
    attempts_dir = run_dir / 'phases' / phase_name / 'attempts'
    completed = []
    for manifest_path in sorted(attempts_dir.glob('A*/phase_manifest.json')):
        payload = json.loads(manifest_path.read_text(encoding='utf-8'))
        if payload.get('status') == 'completed':
            completed.append(int(payload['attempt']))
    if not completed:
        raise RuntimeError(f'No completed attempt for {phase_name}')
    return max(completed)


def read_vectors(path: Path):
    import pandas as pd

    frame = pd.read_csv(path)
    for column in ('embedding', 'fallback_embedding'):
        if column in frame.columns:
            frame[column] = frame[column].map(
                lambda value: json.loads(value) if isinstance(value, str) else value
            )
    return frame


preflight = {
    'execute_stage': EXECUTE_STAGE,
    'run_dir_resolved': str(RUN_DIR),
    'input_mode': (
        'external_csv_override'
        if PROBES_CSV_VALUE or TEMPLATES_CSV_VALUE
        else 'db_protocol_auto'
    ),
    'probes_override_supplied': bool(PROBES_CSV_VALUE),
    'templates_override_supplied': bool(TEMPLATES_CSV_VALUE),
}
preflight


## Execute and record

기본 모드는 `db_protocol_auto`입니다. 자동 모드에서는 전체 gallery template을 사용하므로 exhaustive scope만 전역 인증으로 기록합니다. 얼굴 추출 실패 행은 조용히 숨기지 않고 coverage JSON에 image ID와 probe 유형별 원래/유효 분모를 저장합니다.

`RONBUN_PROBES_CSV`와 `RONBUN_TEMPLATES_CSV`를 **둘 다** 지정하면 기존 외부 CSV override를 사용할 수 있습니다. `candidate_scope=candidate_set` 결과는 전체 gallery에 대한 global certification으로 주장하지 않습니다.


In [ ]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    PROGRESS.emit('실행 시작', expected='LFW 기준 약 1~10분; 입력 크기에 비례')
    import pandas as pd

    from research.compression import ORIGIN_512, PCA_256, PCACompressor
    from research.database import create_database_engine, load_database_settings
    from research.experiments import (
        build_lfw_certification_inputs,
        write_vector_frame_csv,
    )
    from research.search.open_set import (
        build_certified_search_features,
        summarize_certified_search_features,
    )

    with PROGRESS.step('run/input 및 00·02·03 artifact 검증', expected='10초 미만'):
        run, run_manifest = attach_run(RUN_DIR)
        run.verify_inputs()
        run.verify_phase_artifacts('00_protocol_and_run_freeze')
        run.verify_phase_artifacts('02_compressor_fit')
        run.verify_phase_artifacts('03_compressed_materialization_and_index')
        config = run_manifest['config']
        search_cfg = config.get('search', {})
        compression_profile = str(search_cfg.get('compression_profile', PCA_256))
        if compression_profile not in {ORIGIN_512, PCA_256}:
            raise ValueError(
                '04 certification supports origin_512 or pca_256; '
                'PQ is not directly pgvector-searchable.'
            )

    probes_path = resolve_input(PROBES_CSV_VALUE, search_cfg.get('probes_path'))
    templates_path = resolve_input(TEMPLATES_CSV_VALUE, search_cfg.get('templates_path'))
    if (probes_path is None) != (templates_path is None):
        raise ValueError(
            'Provide both RONBUN_PROBES_CSV and RONBUN_TEMPLATES_CSV, or neither.'
        )

    if probes_path is not None:
        if not probes_path.is_file() or not templates_path.is_file():
            raise FileNotFoundError(
                {'probes': str(probes_path), 'templates': str(templates_path)}
            )
        registered_inputs = json.loads(
            run.manifest_path.read_text(encoding='utf-8')
        ).get('inputs', [])
        for role, path in (
            ('probe_vectors', probes_path),
            ('template_vectors', templates_path),
        ):
            same_role = [entry for entry in registered_inputs if entry.get('role') == role]
            if same_role and all(
                Path(entry['path']).resolve() != path.resolve() for entry in same_role
            ):
                raise ValueError(
                    f'{role} already points to another file; start a new run from notebook 00.'
                )
            if not same_role:
                run.record_input(path, role=role)
        run.verify_inputs(roles={'probe_vectors', 'template_vectors'})

    with run.phase('04_probe_search_and_certification') as phase:
        suffix = f'A{phase.attempt:03d}'
        coverage = None
        generated_sources = []
        if probes_path is not None:
            input_mode = 'external_csv_override'
            certificate_space = 'external_declared_space'
            with PROGRESS.step('외부 probe/template CSV 로딩', expected='10초~1분'):
                probes = read_vectors(probes_path)
                templates = read_vectors(templates_path)
        else:
            input_mode = 'db_protocol_auto'
            scope_requested = str(search_cfg.get('candidate_scope', 'exhaustive'))
            if scope_requested != 'exhaustive':
                raise ValueError(
                    'Automatic protocol mode uses the complete gallery and requires '
                    'candidate_scope=exhaustive.'
                )
            protocol_attempt = latest_completed_attempt(
                RUN_DIR, '00_protocol_and_run_freeze'
            )
            protocol_suffix = f'A{protocol_attempt:03d}'
            protocol_dir = RUN_DIR / 'artifacts' / '00_protocol_and_run_freeze'
            protocol_frames = {
                role: pd.read_csv(protocol_dir / f'{role}_{protocol_suffix}.csv')
                for role in (
                    'gallery',
                    'registered_probes',
                    'known_unknown_probes',
                    'unknown_unknown_probes',
                )
            }
            pca = None
            if compression_profile == PCA_256:
                compressor_attempt = latest_completed_attempt(RUN_DIR, '02_compressor_fit')
                compressor_suffix = f'A{compressor_attempt:03d}'
                pca_path = (
                    RUN_DIR
                    / 'artifacts'
                    / '02_compressor_fit'
                    / f'pca_256_{compressor_suffix}.joblib'
                )
                pca = PCACompressor.load(pca_path)
            with PROGRESS.step(
                'DB 벡터 결합 및 gallery template 자동 생성',
                expected='10초~2분',
            ):
                bundle = build_lfw_certification_inputs(
                    create_database_engine(load_database_settings()),
                    run_uid=run.run_id,
                    protocol_frames=protocol_frames,
                    project_root=PROJECT_ROOT,
                    compression_profile=compression_profile,
                    pca=pca,
                )
                probes = bundle.probes
                templates = bundle.templates
                coverage = bundle.coverage
                certificate_space = bundle.certificate_space
            probes_source = phase.attempt_dir / f'generated_probes_{suffix}.csv'
            templates_source = phase.attempt_dir / f'generated_templates_{suffix}.csv'
            coverage_source = phase.attempt_dir / f'protocol_coverage_{suffix}.json'
            write_vector_frame_csv(probes, probes_source)
            write_vector_frame_csv(templates, templates_source)
            coverage_source.write_text(
                json.dumps(coverage, ensure_ascii=False, indent=2), encoding='utf-8'
            )
            generated_sources.extend(
                [probes_source, templates_source, coverage_source]
            )

        PROGRESS.emit(
            '검색 입력 준비 완료',
            probes=len(probes),
            templates=len(templates),
            input_mode=input_mode,
            certificate_space=certificate_space,
        )
        required_probe_types = {'registered', 'known_unknown', 'unknown_unknown'}
        missing_types = sorted(
            required_probe_types.difference(set(probes['probe_type'].astype(str)))
        )
        if missing_types:
            raise ValueError(f'Missing probe types: {missing_types}')
        scope = str(search_cfg.get('candidate_scope', 'exhaustive'))
        configured_gallery_size = search_cfg.get('gallery_size')
        gallery_size = (
            int(configured_gallery_size)
            if configured_gallery_size is not None
            else len(templates)
        )
        with PROGRESS.step(
            'probe×gallery 검색 및 certification 계산',
            expected='1분 이상; O(probes×gallery)',
        ):
            features = build_certified_search_features(
                probes,
                templates,
                compression_profile=compression_profile,
                threshold=float(config['certification']['threshold']),
                top_k=int(search_cfg.get('top_k', 2)),
                candidate_scope=scope,
                gallery_size=gallery_size,
            )
        features['input_mode'] = input_mode
        features['certificate_space'] = certificate_space
        summary = summarize_certified_search_features(features)
        summary.update(
            {
                'input_mode': input_mode,
                'compression_profile': compression_profile,
                'certificate_space': certificate_space,
                'protocol_coverage': coverage,
            }
        )
        PROGRESS.emit('검색 완료, artifact 저장 시작', certified_rows=len(features))
        features_source = phase.attempt_dir / f'certified_features_{suffix}.csv'
        summary_source = phase.attempt_dir / f'certification_summary_{suffix}.json'
        write_vector_frame_csv(features, features_source)
        summary_source.write_text(
            json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        for source in [*generated_sources, features_source, summary_source]:
            phase.publish_artifact(source)
        phase.record_counts(
            probes=len(probes),
            templates=len(templates),
            certified_rows=len(features),
        )
        phase.record(
            'certification_scope',
            candidate_scope=scope,
            gallery_size=gallery_size,
            global_claim=bool(features['certification_global_claim'].all()),
            input_mode=input_mode,
            certificate_space=certificate_space,
        )
    PROGRESS.emit(
        '04 완료',
        probes=len(probes),
        templates=len(templates),
        certified_rows=len(features),
    )
    result = {
        'status': 'completed',
        'run_id': run.run_id,
        'candidate_scope': scope,
        'gallery_size': gallery_size,
        'input_mode': input_mode,
        'certificate_space': certificate_space,
        **summary,
    }
else:
    PROGRESS.emit(
        '검토 모드 완료: 검색과 certification을 실행하지 않음', expected='1초 미만'
    )
result


## Next step

`protocol_coverage`에서 probe 유형별 원래/유효 분모와 누락 image ID를 먼저 확인합니다. 그다음 certification coverage, defer rate, exact fallback rate를 확인한 뒤 05로 이동합니다. PCA의 `certificate_space=pca_reconstructed_512` 결과를 256D pgvector retrieval 결과와 동일한 것으로 표현하지 않습니다.
